In [3]:
import os
import numpy as np
import glob

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def process_case(case_folder, path_to_dataset, pattern_list, output_dir, subject_string, progress_bar):
    """
    Processes each case folder, copying files matching the patterns to the output directory.
    """
    # Define the full path to the current case folder
    case_path = os.path.join(path_to_dataset, case_folder)
    case_output_dir = os.path.join(output_dir, case_folder)
    os.makedirs(case_output_dir, exist_ok=True)

    # Loop through each file in the case folder and its subdirectories
    for root, _, files in os.walk(case_path):
        for file in files:
            if any(pattern in file for pattern in pattern_list):
                # Copy or move the file to the organized case folder in the output directory
                src = os.path.join(root, file)
                dst = os.path.join(case_output_dir, file)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy(src, dst)
                
    progress_bar.update(1)

def organize_files_by_case(path_to_dataset, pattern_list, output_dir, subject_string="sub-strokecase", max_workers=4):
    """
    Organizes files matching specific patterns into case-specific folders using multiple workers.

    Parameters:
    - path_to_dataset (str): Path to the main dataset directory containing case folders.
    - pattern_list (list): List of filename patterns to match (e.g., ["FLAIR.nii.gz", "adc"]).
    - output_dir (str): Path to the output directory where organized folders will be created.
    - subject_string (str): Prefix string identifying subject folders (default is "sub-strokecase").
    - max_workers (int): Maximum number of worker threads to use for processing.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Recursively search for all case folders with subject_string in their name
    case_folders = []
    for root, dirs, _ in os.walk(path_to_dataset):
        case_folders.extend([os.path.relpath(os.path.join(root, d), path_to_dataset) 
                             for d in dirs if d.startswith(subject_string)])

    with tqdm(total=len(case_folders), desc="Organizing files") as progress_bar:
        # Use ThreadPoolExecutor to process each case in parallel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(process_case, case_folder, path_to_dataset, pattern_list, output_dir, subject_string, progress_bar)
                for case_folder in case_folders
            ]
            # Wait for all tasks to complete
            for future in futures:
                future.result()

    print("Files organized by case successfully.")

In [ ]:
# ISLES22
organize_files_by_case(
    path_to_dataset='/Volumes/Kurtlab/Brats2024/Other_dataset/ISLES 2022/ISLES-2022',
    pattern_list=['FLAIR.nii.gz'],
    output_dir='/Volumes/Kurtlab/Brats2024/Other_dataset/ISLES 2022/v2_OrganizedISLES22',
    max_workers=8
)

In [4]:
#ATLAS
organize_files_by_case(
    path_to_dataset='/Volumes/Kurtlab/Brats2024/Other_dataset/ATLAS_2',
    pattern_list=['FLAIR.nii.gz', 'T1w.nii.gz'],
    output_dir='/Volumes/Kurtlab/Brats2024/Other_dataset/ATLAS_2_cleaned',
    subject_string="sub-",
    max_workers=8
)

Organizing files: 0it [00:00, ?it/s]

Files organized by case successfully.
